In [12]:
# 1. Initialization

import matplotlib.pyplot as plt
import importlib
from torch.utils.data import (
    Dataset,
    DataLoader,
    Subset,
    TensorDataset,
    random_split
)
import os
import pandas as pd
import numpy as np
from pathlib import Path
from torchvision import datasets
from torchvision.transforms import v2
import torch
from torch import nn

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    else:
        return torch.device("cpu")

device = get_device()

print('torch:', torch.__version__)
print('built CUDA:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print(f"Selected device: {device}")

torch: 2.13.0+cu130
built CUDA: 13.0
CUDA available: True
device: NVIDIA GeForce RTX 5060 Laptop GPU
Selected device: cuda:0


# Model training vs evaluation mode
- `model.train()` and `model.eval()` sets model to training/evaluation mode respectively.
- This matters for modules whose behaviour changes between modes (ex: `Dropout`,`BatchNorm`).

# torch.inference_mode() vs torch.no_grad()
- Main difference is torch.inference_mode() strips additional autograd-related overhead -> Faster performance in general.
- Use `no_grad()` when we need gradient-disabled region, but later use resulting tensors in autograd-tracked computations.

In [4]:
# Loading dataset

training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

split_generator = (torch.Generator().manual_seed(67))

train_subset, val_subset = (
    random_split(
        training_data,
        lengths=[54_000, 6_000],
        generator=split_generator
    )
)

print(len(train_subset))
print(len(val_subset))

54000
6000


In [6]:
train_loader = DataLoader(
    train_subset,
    batch_size=128,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_subset,
    batch_size=128,
    shuffle=False,
    num_workers=0,
)

In [7]:
class FashionMLP(nn.Module):
    def __init__(
        self,
        hidden_features: int = 128,
    ) -> None:
        super().__init__()

        self.flatten = nn.Flatten(
            start_dim=1
        )

        self.network = nn.Sequential(
            nn.Linear(
                28 * 28,
                hidden_features,
            ),
            nn.ReLU(),
            nn.Linear(
                hidden_features,
                10,
            ),
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        x = self.flatten(x)
        return self.network(x)

In [8]:
model = FashionMLP().to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
)

In [20]:
import engine

importlib.reload(engine)

<module 'engine' from '/home/halzyon/Data/coding_work/projects/pytorch-lab/notebook/engine.py'>

In [21]:
train_metrics = engine.train_one_epoch(
    model=model,
    dataloader=train_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device,
)

val_metrics = engine.evaluate(
    model=model,
    dataloader=val_loader,
    loss_fn=loss_fn,
    device=device,
)

In [22]:
print("train:", train_metrics)
print("validation:", val_metrics)

train: {'loss': 0.5063761276139154, 'accuracy': 0.8220185185185185}
validation: {'loss': 0.5336239658991496, 'accuracy': 0.7928333333333333}
